# HW2 – Automatic Speech Recognition

**Student name:** _Your Name_  
**Student ID:** _Your ID_  
**Date:** _YYYY-MM-DD_

## 0. Setup

In [ ]:
import torch
import librosa
import numpy as np
import IPython.display as ipd
from transformers import pipeline

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

## 1. Load Audio

In [ ]:
AUDIO_PATH = "sample.wav"  # TODO: update path if needed
REFERENCE  = "the quick brown fox jumps over the lazy dog"  # TODO: provide ground-truth text

y, sr = librosa.load(AUDIO_PATH, sr=16000)  # models expect 16 kHz
print(f"Sample rate : {sr} Hz")
print(f"Duration    : {len(y)/sr:.2f} s")
ipd.Audio(y, rate=sr)

## 2. Transcription with a Pre-trained Model

We use the Hugging Face `pipeline` API to transcribe the audio with a pre-trained model.

In [ ]:
# TODO: try different model checkpoints, e.g.:
#   "openai/whisper-tiny"  – fast, lower accuracy
#   "openai/whisper-base"  – balanced
#   "facebook/wav2vec2-base-960h" – CTC-based

MODEL_NAME = "openai/whisper-tiny"

asr = pipeline("automatic-speech-recognition", model=MODEL_NAME)
result = asr({"array": y, "sampling_rate": sr})

hypothesis = result["text"].strip().lower()
print("Hypothesis :", hypothesis)
print("Reference  :", REFERENCE)

## 3. Word Error Rate (WER) Evaluation

In [ ]:
def compute_wer(reference: str, hypothesis: str) -> float:
    """Compute Word Error Rate (WER) between reference and hypothesis strings."""
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    r, h = len(ref_words), len(hyp_words)

    # Dynamic programming
    dp = np.zeros((r + 1, h + 1), dtype=int)
    for i in range(r + 1):
        dp[i][0] = i
    for j in range(h + 1):
        dp[0][j] = j
    for i in range(1, r + 1):
        for j in range(1, h + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[r][h] / r if r > 0 else 0.0

wer = compute_wer(REFERENCE, hypothesis)
print(f"WER: {wer:.2%}")

## 4. Experiments

Try at least **two** different model checkpoints and compare their WER on the same audio.

In [ ]:
# TODO: fill in the models you want to compare
MODELS = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    # "facebook/wav2vec2-base-960h",
]

results = []
for model_name in MODELS:
    asr_model = pipeline("automatic-speech-recognition", model=model_name)
    hyp = asr_model({"array": y, "sampling_rate": sr})["text"].strip().lower()
    w = compute_wer(REFERENCE, hyp)
    results.append({"model": model_name, "hypothesis": hyp, "WER": f"{w:.2%}"})
    print(f"{model_name}: WER = {w:.2%}")

print("\nSummary:")
for r in results:
    print(r)

## 5. Analysis & Questions

**Q1.** Which model achieved the lowest WER? What might explain the difference?

_Your answer here._

---

**Q2.** What types of errors (substitution, insertion, deletion) dominate? Why?

_Your answer here._

---

**Q3.** What pre-processing steps could you apply to the audio to potentially reduce WER?

_Your answer here._